# Simulated dataset generation

Generates a fully reproducible synthetic daily dataset in the **wide** format the
framework consumes (index `ds` = daily dates, columns = individual time series).

Target: **1000 series**, **10 years** of daily history each.

Each series is built from components that AutoETS / AutoARIMA / Prophet can all
model well:
- a positive **level** with a (possibly damped) **linear trend**,
- **weekly** seasonality (all series) and **yearly** seasonality (a subset),
- **AR(1)** autocorrelated noise (ARIMA's sweet spot),
- multiplicative or additive noise depending on the archetype.

Everything derives from a single master `SEED`, so re-running reproduces the
exact same CSV. The output goes to `data/simulated/simulated.csv`.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..')  # Run from repo root

import numpy as np
import pandas as pd

In [ ]:
# ---- Reproducibility: one master seed drives everything ----
SEED = 20240913

N_SERIES = 1000
N_YEARS = 10
N_DAYS = 365 * N_YEARS + N_YEARS // 4  # ~10 years including leap days => 3652
START_DATE = pd.Timestamp('2014-01-06')  # a Monday, so weekly phase is clean

OUTPUT_DIR = pathlib.Path('data/simulated')
OUTPUT_PATH = OUTPUT_DIR / 'simulated.csv'

master_rng = np.random.default_rng(SEED)
# Give every series its own independent, deterministic seed derived from the master.
series_seeds = master_rng.integers(0, 2**32 - 1, size=N_SERIES)

dates = pd.date_range(start=START_DATE, periods=N_DAYS, freq='D')
print(f'{N_SERIES} series x {N_DAYS} days, {dates[0].date()} -> {dates[-1].date()}')

## Per-series generator

Each series is drawn from one of a few **archetypes** so the pool is heterogeneous
(different trend/seasonality/noise mixes) while every series stays smooth and
seasonal enough to be genuinely forecastable by all three models. All randomness
is drawn from a per-series `default_rng(seed)`, so a series depends only on its own
seed.

In [ ]:
def _fourier(t, period, n_harmonics, rng, amp):
    """Sum of sine/cosine harmonics with random phases/amplitudes.
    Returns a zero-mean seasonal signal (relative units)."""
    out = np.zeros_like(t, dtype=float)
    for k in range(1, n_harmonics + 1):
        a = rng.normal(0, amp / k)
        b = rng.normal(0, amp / k)
        out += a * np.sin(2 * np.pi * k * t / period) + b * np.cos(2 * np.pi * k * t / period)
    return out


def _ar1(n, phi, sigma, rng):
    """Stationary AR(1) noise with lag-1 coefficient phi and innovation sd sigma."""
    e = rng.normal(0, sigma, size=n)
    x = np.empty(n)
    x[0] = e[0] / np.sqrt(max(1e-6, 1 - phi**2))  # stationary start
    for i in range(1, n):
        x[i] = phi * x[i - 1] + e[i]
    return x


def _random_walk(n, sigma, rng):
    """Zero-start Gaussian random walk (integrated noise): w_t = w_{t-1} + eta_t.
    A unit-root / stochastic-trend component whose uncertainty COMPOUNDS with horizon
    and does NOT average out under 7-day aggregation -- this is what injects genuine,
    unbiased forecast headroom at the TOP (weekly) level."""
    if sigma <= 0:
        return np.zeros(n)
    return np.cumsum(rng.normal(0, sigma, size=n))


# Amplitude of the geometric random walk on the level (per-day log-return sd).
# 0.0 recovers the previous purely-seasonal generator. ~0.01-0.03 makes the weekly
# aggregate genuinely uncertain without letting the walk swamp the seasonal signal.
RW_SIGMA = 0.06


# Archetypes: (name, weight). Weights need not sum to 1.
ARCHETYPES = [
    ('trend_weekly',        0.30),  # trend + weekly seasonality, multiplicative noise
    ('trend_weekly_yearly', 0.30),  # + yearly seasonality
    ('damped_trend',        0.20),  # damped/plateauing trend (ETS damped)
    ('stationary_ar',       0.20),  # flat level, strong AR(1), weekly season (ARIMA)
]
_arch_names = [a for a, _ in ARCHETYPES]
_arch_probs = np.array([w for _, w in ARCHETYPES])
_arch_probs = _arch_probs / _arch_probs.sum()


def generate_series(seed: int, n: int) -> np.ndarray:
    """Generate one positive-valued daily series of length n from its own seed."""
    rng = np.random.default_rng(seed)
    t = np.arange(n, dtype=float)

    archetype = rng.choice(_arch_names, p=_arch_probs)

    # Base level spans several orders of magnitude across series (heterogeneous scale).
    base_level = float(10 ** rng.uniform(1.0, 4.0))  # ~10 .. 10000

    # --- Trend ---
    # Relative annual growth; damped archetype saturates over time.
    annual_growth = rng.normal(0.0, 0.15)  # +/- ~15% per year typical
    years = t / 365.0
    if archetype == 'damped_trend':
        phi_damp = rng.uniform(0.90, 0.99)
        # Geometric damping of the yearly increment -> plateau
        damped_years = (1 - phi_damp ** (years + 1)) / (1 - phi_damp)
        trend = annual_growth * damped_years
    elif archetype == 'stationary_ar':
        trend = np.zeros_like(t)
    else:
        trend = annual_growth * years
    trend_factor = 1.0 + trend  # multiplicative around the level
    trend_factor = np.clip(trend_factor, 0.05, None)

    # --- Weekly seasonality (all series) ---
    weekly_amp = rng.uniform(0.05, 0.35)
    weekly = _fourier(t, period=7, n_harmonics=3, rng=rng, amp=weekly_amp)

    # --- Yearly seasonality (subset) ---
    if archetype == 'trend_weekly_yearly':
        yearly_amp = rng.uniform(0.10, 0.40)
        yearly = _fourier(t, period=365.25, n_harmonics=5, rng=rng, amp=yearly_amp)
    else:
        yearly = np.zeros_like(t)

    seasonal_factor = 1.0 + weekly + yearly
    seasonal_factor = np.clip(seasonal_factor, 0.05, None)

    # --- AR(1) noise ---
    if archetype == 'stationary_ar':
        phi = rng.uniform(0.5, 0.85)
        noise_sd = rng.uniform(0.10, 0.25)
    else:
        phi = rng.uniform(0.1, 0.5)
        noise_sd = rng.uniform(0.03, 0.12)
    ar = _ar1(n, phi=phi, sigma=noise_sd, rng=rng)
    noise_factor = np.exp(ar)  # multiplicative, keeps values positive

    # --- Stochastic trend: geometric random walk on the level (all series) ---
    # exp(cumsum of small shocks) -> multiplicative, strictly positive, unit-root.
    # Survives 7-day aggregation, so it creates real top-level forecast headroom.
    walk_factor = np.exp(_random_walk(n, sigma=RW_SIGMA, rng=rng))

    series = base_level * trend_factor * seasonal_factor * walk_factor * noise_factor

    # Guard against non-finite/negative artifacts.
    series = np.where(np.isfinite(series), series, base_level)
    series = np.clip(series, 1e-3, None)
    return series.astype('float32')

In [ ]:
# ---- Generate all series ----
data = {f'S{idx:04d}': generate_series(int(series_seeds[idx]), N_DAYS)
        for idx in range(N_SERIES)}

X = pd.DataFrame(data, index=dates)
X.index.name = 'ds'

assert X.shape == (N_DAYS, N_SERIES)
assert np.isfinite(X.to_numpy()).all()
assert (X.to_numpy() > 0).all()
X.iloc[:3, :5]

In [ ]:
# ---- Quick sanity: scale spread and a couple of example series ----
print('per-series mean level percentiles:')
print(X.mean().describe(percentiles=[.1, .5, .9]).to_string())

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(13, 6))
for ax, col in zip(axes.ravel(), ['S0000', 'S0001', 'S0002', 'S0003']):
    X[col].plot(ax=ax, lw=0.6)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Serialize to the framework's wide CSV format ----
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
X.to_csv(OUTPUT_PATH)
print(f'wrote {OUTPUT_PATH}  ({X.shape[0]} rows x {X.shape[1]} cols)')

# Round-trip check: reload the way utils would (index_col='ds', parse dates).
reloaded = pd.read_csv(OUTPUT_PATH, index_col='ds', parse_dates=True)
assert reloaded.shape == X.shape
assert list(reloaded.columns) == list(X.columns)
assert np.allclose(reloaded.to_numpy(), X.to_numpy(), rtol=1e-5, atol=1e-5)
print('round-trip OK')